# Tarea 2 — EDA y Series de Tiempo
## EO4040 Introducción a la Ciencia de Datos para Economía
**Tecnológico de Monterrey — Gobierno y Transformación Pública**

**Valor total: 80 puntos automáticos + 20 puntos presentación = 100 puntos**

**Instrucciones:**
- Ejecuta las celdas en orden.
- Reemplaza `...` con tu código.
- No modifiques las celdas de tests ni las de configuración.
- Preguntas 1–4: Análisis Exploratorio de Datos (sesión 3).
- Preguntas 5–8: Series de Tiempo (sesión 4).
- Al terminar, descarga tu notebook y entrégalo en Canvas.

In [ ]:
# Instalaciones necesarias (ejecuta primero, ~1-2 min en Colab)
import subprocess, sys
for pkg in ["wbgapi", "statsmodels", "otter-grader==7.0.0"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
print("Instalaciones completas.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.stattools import durbin_watson
from statsmodels.tsa.stattools import acf as sm_acf
import warnings
warnings.filterwarnings("ignore")

import otter
grader = otter.Notebook("tarea2_master.ipynb")

np.random.seed(42)
print("Importaciones listas.")

---
## Parte 1: Análisis Exploratorio de Datos (Sesión 3)

Indicadores del **Banco Mundial (WDI)** para 27 países:
- **19 países de América Latina:** ARG, BOL, BRA, CHL, COL, CRI, CUB, DOM, ECU, GTM, HND, MEX, NIC, PAN, PER, PRY, SLV, URY, VEN
- **8 países de referencia:** USA, DEU, NOR, KOR, ZAF, NGA, IND, CHN

| Código WDI | Columna |
|---|---|
| NY.GDP.PCAP.PP.CD | pib_pc_ppp |
| SI.POV.GINI | gini |
| SL.UEM.TOTL.ZS | desempleo |
| SE.XPD.TOTL.GD.ZS | gasto_educacion |
| SL.TLF.CACT.FE.ZS | participacion_fem |

In [ ]:
# Esta celda NO se modifica — carga los datos (API o fixture local).
import os, pathlib
paises_latam = [
    "ARG","BOL","BRA","CHL","COL","CRI","CUB","DOM",
    "ECU","GTM","HND","MEX","NIC","PAN","PER","PRY",
    "SLV","URY","VEN",
]
paises_ref   = ["USA","DEU","NOR","KOR","ZAF","NGA","IND","CHN"]
todos_paises = paises_latam + paises_ref

_fixture_wb = pathlib.Path("fixture_wb.csv")
_loaded_wb  = False
if _fixture_wb.exists():
    df_wb = pd.read_csv(_fixture_wb)
    _loaded_wb = True
    print("Datos cargados desde fixture local.")
else:
    try:
        import wbgapi as wb
        indicadores = {
            "NY.GDP.PCAP.PP.CD": "pib_pc_ppp",
            "SI.POV.GINI":        "gini",
            "SL.UEM.TOTL.ZS":     "desempleo",
            "SE.XPD.TOTL.GD.ZS":  "gasto_educacion",
            "SL.TLF.CACT.FE.ZS":  "participacion_fem",
        }
        raw_wb = wb.data.DataFrame(
            list(indicadores.keys()),
            economy=todos_paises,
            time=range(2018, 2024),
            mrv=1,
            skipBlanks=False,
            labels=False,
        )
        df_wb = raw_wb.reset_index()
        rename_map = {k: v for k, v in indicadores.items() if k in df_wb.columns}
        df_wb = df_wb.rename(columns=rename_map)
        if "economy" in df_wb.columns:
            df_wb = df_wb.rename(columns={"economy": "pais"})
        df_wb["region"] = df_wb["pais"].apply(
            lambda c: "latam" if c in paises_latam else "referencia"
        )
        for col in indicadores.values():
            if col not in df_wb.columns:
                df_wb[col] = float("nan")
        cols_keep = ["pais","region"] + list(indicadores.values())
        df_wb = df_wb[cols_keep].reset_index(drop=True)
        _loaded_wb = True
        print("Datos descargados desde API.")
    except Exception as _e:
        raise RuntimeError(f"No se pudo cargar df_wb: {_e}")
print("Filas:", len(df_wb), "  Columnas:", list(df_wb.columns))
df_wb.head()

---
### Pregunta 1 (10 pts) — Análisis univariado categórico

Calcula la **tabla de frecuencias** de la columna `region` en `df_wb`.

Guarda:
- `tabla_freq_region`: `pd.Series` con conteos por categoría, ordenado de mayor a menor.
- `moda_region`: categoría con mayor frecuencia (string).

In [ ]:
# BEGIN SOLUTION
tabla_freq_region = df_wb["region"].value_counts().sort_values(ascending=False)
moda_region = tabla_freq_region.index[0]
# END SOLUTION

print("Tabla de frecuencias:")
print(tabla_freq_region)
print("Moda:", moda_region)

In [ ]:
assert isinstance(tabla_freq_region, pd.Series), "tabla_freq_region debe ser pd.Series"
assert len(tabla_freq_region) == 2, "Debe haber 2 categorias"
assert tabla_freq_region["latam"] == 19, "latam debe tener 19 paises"
assert tabla_freq_region["referencia"] == 8, "referencia debe tener 8 paises"
assert moda_region == "latam", "la moda debe ser latam"

In [ ]:
grader.check("q1")

---
### Pregunta 2 (10 pts) — Análisis univariado continuo

Para los países de `region == 'latam'`, calcula estadísticas de `pib_pc_ppp`.

Guarda:
- `desc_pib`: `pd.Series` con claves exactas `'media'`, `'mediana'`, `'desv_std'`, `'minimo'`, `'maximo'`.
- `asimetria_pib`: coeficiente de asimetría de Pearson (float, usa `scipy.stats.skew`).

In [ ]:
# BEGIN SOLUTION
pib_latam = df_wb.loc[df_wb["region"] == "latam", "pib_pc_ppp"].dropna()

desc_pib = pd.Series({
    "media":    pib_latam.mean(),
    "mediana":  pib_latam.median(),
    "desv_std": pib_latam.std(),
    "minimo":   pib_latam.min(),
    "maximo":   pib_latam.max(),
})

asimetria_pib = float(stats.skew(pib_latam))
# END SOLUTION

print("Estadisticas PIB pc PPP (latam):")
print(desc_pib.round(2))
print("Asimetria:", round(asimetria_pib, 4))

In [ ]:
assert isinstance(desc_pib, pd.Series), "desc_pib debe ser pd.Series"
assert list(desc_pib.index) == ["media","mediana","desv_std","minimo","maximo"], "Las claves deben ser ['media','mediana','desv_std','minimo','maximo']"
assert desc_pib["media"] > desc_pib["mediana"], "Se espera media > mediana"
assert isinstance(asimetria_pib, float), "asimetria_pib debe ser float"
assert asimetria_pib > 0, "Se espera asimetria positiva"
assert desc_pib["minimo"] < desc_pib["mediana"] < desc_pib["maximo"]

In [ ]:
grader.check("q2")

---
### Pregunta 3 (10 pts) — Correlación bivariada

Calcula la **matriz de correlación de Pearson** para las 5 variables continuas de `df_wb`.

Guarda:
- `corr_matrix`: `pd.DataFrame` 5×5 con la matriz de correlación.
- `par_mayor_corr`: tupla `(var1, var2)` con el par de mayor correlación en valor absoluto (excluye diagonal), en **orden alfabético**.

In [ ]:
# BEGIN SOLUTION
cols_continuas = ["pib_pc_ppp","gini","desempleo","gasto_educacion","participacion_fem"]
# Incluir solo columnas con al menos 2 valores no-NaN para que corr() sea valida
cols_con_datos = [c for c in cols_continuas if df_wb[c].notna().sum() >= 2]
corr_matrix = df_wb[cols_continuas].corr(method="pearson")

mask = np.ones(corr_matrix.shape, dtype=bool)
np.fill_diagonal(mask, False)
# Ignorar NaN al buscar el par de mayor correlacion
par = corr_matrix.abs().where(mask, np.nan).stack().idxmax()
par_mayor_corr = tuple(sorted(par))
# END SOLUTION

print("Matriz de correlacion:")
print(corr_matrix.round(3))
print("Par con mayor correlacion:", par_mayor_corr)

In [ ]:
assert isinstance(corr_matrix, pd.DataFrame), "corr_matrix debe ser pd.DataFrame"
assert corr_matrix.shape == (5, 5), "corr_matrix debe ser 5x5"
# La diagonal es 1.0 donde hay datos; puede ser NaN si toda la columna es NaN
_diag = np.diag(corr_matrix.to_numpy().astype(float))
assert np.all((_diag == 1.0) | np.isnan(_diag)), "La diagonal debe ser 1.0 o NaN"
_vals = corr_matrix.to_numpy().astype(float)
assert np.nanmax(_vals) <= 1.0 + 1e-9
assert np.nanmin(_vals) >= -1.0 - 1e-9
_cols = {"pib_pc_ppp","gini","desempleo","gasto_educacion","participacion_fem"}
assert isinstance(par_mayor_corr, tuple) and len(par_mayor_corr) == 2
assert par_mayor_corr[0] in _cols and par_mayor_corr[1] in _cols
assert par_mayor_corr[0] != par_mayor_corr[1]
assert par_mayor_corr[0] < par_mayor_corr[1], "Debe estar en orden alfabetico"

In [ ]:
grader.check("q3")

---
### Pregunta 4 (10 pts) — Estadísticas por grupo

Compara la **mediana del desempleo** entre regiones.

Guarda:
- `mediana_desempleo_grupo`: `pd.Series` con mediana de `desempleo` por `region`, ordenado de mayor a menor.
- `region_mayor_desempleo`: región con mayor mediana (string).
- `diferencia_medianas`: diferencia `latam - referencia` (float).

In [ ]:
# BEGIN SOLUTION
mediana_desempleo_grupo = (
    df_wb.groupby("region")["desempleo"]
    .median()
    .sort_values(ascending=False)
)
region_mayor_desempleo = mediana_desempleo_grupo.index[0]
diferencia_medianas = float(
    mediana_desempleo_grupo["latam"] - mediana_desempleo_grupo["referencia"]
)
# END SOLUTION

print("Mediana desempleo por region:")
print(mediana_desempleo_grupo.round(2))
print("Region mayor desempleo:", region_mayor_desempleo)
print("Diferencia (latam - referencia):", round(diferencia_medianas, 2))

In [ ]:
assert isinstance(mediana_desempleo_grupo, pd.Series)
assert len(mediana_desempleo_grupo) == 2
assert set(mediana_desempleo_grupo.index) == {"latam","referencia"}
assert isinstance(region_mayor_desempleo, str)
_vals = list(mediana_desempleo_grupo.values)
assert _vals[0] >= _vals[1], "Debe estar ordenado de mayor a menor"
assert isinstance(diferencia_medianas, float)
_exp = float(mediana_desempleo_grupo["latam"] - mediana_desempleo_grupo["referencia"])
assert abs(diferencia_medianas - _exp) < 0.001

In [ ]:
grader.check("q4")

---
## Parte 2: Series de Tiempo (Sesión 4)

Inflación anual del **Banco Mundial** para COL, DOM, HND:
- **Indicador:** `FP.CPI.TOTL.ZG` — Inflación IPC (% anual)
- **Periodo:** 2000–2023
- **Fuente:** wbgapi — no se necesita descargar archivos externos.

In [ ]:
# Esta celda NO se modifica — carga los datos (API o fixture local).
_fixture_inf = pathlib.Path("fixture_inf.csv")
if _fixture_inf.exists():
    df_inf_raw = pd.read_csv(_fixture_inf)
    df_inf_raw["anio"] = df_inf_raw["anio"].astype(int)
    df_inf_raw = df_inf_raw.sort_values(["pais","anio"]).reset_index(drop=True)
    print("Datos inflacion cargados desde fixture local.")
else:
    try:
        import wbgapi as wb
        paises_st = ["COL", "DOM", "HND"]
        raw_inf = wb.data.DataFrame(
            "FP.CPI.TOTL.ZG",
            economy=paises_st,
            time=range(2000, 2024),
            skipBlanks=False,
            labels=False,
        )
        # wbgapi devuelve economias en filas, anios en columnas (YR2000...)
        # Hay que transponer a formato long: pais, anio, inflacion
        df_inf_raw = (
            raw_inf
            .T  # anios en filas, paises en columnas
            .reset_index()
            .melt(id_vars="index", var_name="pais", value_name="inflacion")
            .rename(columns={"index": "anio"})
        )
        df_inf_raw["anio"] = df_inf_raw["anio"].astype(str).str.extract(r"(\d{4})").astype(int)
        df_inf_raw = df_inf_raw.dropna(subset=["anio"]).sort_values(["pais","anio"]).reset_index(drop=True)
        print("Datos inflacion descargados desde API.")
    except Exception as _e:
        raise RuntimeError(f"No se pudo cargar df_inf_raw: {_e}")
print("Forma:", df_inf_raw.shape, "  Paises:", df_inf_raw['pais'].unique())
df_inf_raw.head(8)

---
### Pregunta 5 (10 pts) — Preparación de series de tiempo

Crea una serie de tiempo para **Colombia (COL)**.

Guarda:
- `serie_col`: `pd.Series` con inflación anual de COL, índice `pd.PeriodIndex` frecuencia `'Y'`, nombre `'inflacion_COL'`.
- `n_obs_col`: número de observaciones no nulas (int).
- `anio_max_inflacion_col`: año con mayor inflación (int).

In [ ]:
# BEGIN SOLUTION
df_col = df_inf_raw[df_inf_raw["pais"] == "COL"].copy()

serie_col = pd.Series(
    df_col["inflacion"].values,
    index=pd.PeriodIndex(df_col["anio"].values, freq="Y"),
    name="inflacion_COL",
)

n_obs_col = int(serie_col.dropna().count())
anio_max_inflacion_col = int(serie_col.idxmax().year)
# END SOLUTION

print("Serie COL (primeros 5):")
print(serie_col.head())
print("Obs no nulas:", n_obs_col)
print("Anio max inflacion:", anio_max_inflacion_col)

In [ ]:
assert isinstance(serie_col, pd.Series)
assert isinstance(serie_col.index, pd.PeriodIndex), "El indice debe ser PeriodIndex"
assert str(serie_col.index.freqstr).startswith(("A","Y")), "Frecuencia debe ser anual"
assert serie_col.name == "inflacion_COL"
assert len(serie_col) == 24, "Debe tener 24 observaciones (2000-2023)"
assert isinstance(n_obs_col, int) and 20 <= n_obs_col <= 24
assert isinstance(anio_max_inflacion_col, int)
assert 2000 <= anio_max_inflacion_col <= 2023

In [ ]:
grader.check("q5")

---
### Pregunta 6 (10 pts) — Descomposición de serie de tiempo

> **Nota técnica:** Los datos son anuales. Usamos `period=2` para extraer tendencia de mediano plazo y residuos. Interpola los NaN antes de descomponer.

Aplica `seasonal_decompose` a `serie_col` con `model='additive'`, `period=2`, `extrapolate_trend='freq'`.

Guarda:
- `descomp_col`: resultado de `seasonal_decompose`.
- `tendencia_col`: `pd.Series` con el componente de tendencia, sin NaN.
- `residuos_col`: `pd.Series` con el componente de residuos, sin NaN.

In [ ]:
# BEGIN SOLUTION
serie_col_interp = serie_col.interpolate(method="linear")
descomp_col = seasonal_decompose(
    serie_col_interp,
    model="additive",
    period=2,
    extrapolate_trend="freq",
)
tendencia_col = descomp_col.trend.dropna()
residuos_col  = descomp_col.resid.dropna()
# END SOLUTION

print("Longitud tendencia:", len(tendencia_col))
print("Longitud residuos: ", len(residuos_col))
print("Tendencia (primeros 5):")
print(tendencia_col.head().round(4))

In [ ]:
assert hasattr(descomp_col, "trend")
assert hasattr(descomp_col, "seasonal")
assert hasattr(descomp_col, "resid")
assert isinstance(tendencia_col, pd.Series)
assert tendencia_col.isna().sum() == 0, "tendencia_col no debe tener NaN"
assert isinstance(residuos_col, pd.Series)
assert residuos_col.isna().sum() == 0, "residuos_col no debe tener NaN"
assert len(tendencia_col) > 0
assert np.isfinite(tendencia_col.values).all()

In [ ]:
grader.check("q6")

---
### Pregunta 7 (10 pts) — Función de Autocorrelación (ACF)

Calcula la ACF de `serie_col` (sin NaN) hasta el **lag 5**.

Guarda:
- `acf_col`: `np.ndarray` con ACF para lags 0–5 (longitud 6). Usa `sm_acf(serie_col_clean, nlags=5, fft=False)` donde `serie_col_clean = serie_col.dropna()`.
- `lag_mayor_acf`: lag (int) con mayor ACF, **excluyendo lag 0**.

In [ ]:
# BEGIN SOLUTION
serie_col_clean = serie_col.dropna()
acf_col = sm_acf(serie_col_clean, nlags=5, fft=False)
lag_mayor_acf = int(np.argmax(acf_col[1:]) + 1)
# END SOLUTION

print("ACF lags 0-5:")
for i, v in enumerate(acf_col):
    print(f"  lag {i}: {v:.4f}")
print("Lag mayor ACF (excl. lag 0):", lag_mayor_acf)

In [ ]:
assert isinstance(acf_col, np.ndarray), "acf_col debe ser np.ndarray"
assert len(acf_col) == 6, "Debe tener 6 elementos (lags 0-5)"
assert abs(acf_col[0] - 1.0) < 1e-6, "Lag 0 debe ser 1.0"
assert np.all(acf_col >= -1.0 - 1e-9)
assert np.all(acf_col <= 1.0 + 1e-9)
assert isinstance(lag_mayor_acf, int)
assert 1 <= lag_mayor_acf <= 5
assert acf_col[lag_mayor_acf] == max(acf_col[1:])

In [ ]:
grader.check("q7")

---
### Pregunta 8 (10 pts) — Regresión OLS y diagnóstico Durbin-Watson

Ajusta una regresión OLS: Y = inflación COL, X = año (2000–2023). Incluye constante con `sm.add_constant`.

Guarda:
- `modelo_col`: resultado de `sm.OLS(...).fit()`.
- `pendiente_col`: coeficiente de `anio` (float).
- `dw_col`: estadístico Durbin-Watson sobre los residuos (float).
- `hay_autocorr`: `True` si `dw_col < 1.5` (bool).

In [ ]:
# BEGIN SOLUTION
df_col_ols = df_inf_raw[df_inf_raw["pais"] == "COL"].dropna(subset=["inflacion"]).copy()
X = sm.add_constant(df_col_ols["anio"].astype(float))
y = df_col_ols["inflacion"].astype(float)
modelo_col    = sm.OLS(y, X).fit()
pendiente_col = float(modelo_col.params["anio"])
dw_col        = float(durbin_watson(modelo_col.resid))
hay_autocorr  = bool(dw_col < 1.5)
# END SOLUTION

print("Pendiente (tendencia anual):", round(pendiente_col, 4))
print("Durbin-Watson:", round(dw_col, 4))
print("Autocorrelacion positiva:", hay_autocorr)

In [ ]:
assert hasattr(modelo_col, "params")
assert hasattr(modelo_col, "resid")
assert isinstance(pendiente_col, float)
assert pendiente_col < 0, "Se espera pendiente negativa"
assert isinstance(dw_col, float)
assert 0 <= dw_col <= 4
assert isinstance(hay_autocorr, bool)
assert hay_autocorr == (dw_col < 1.5)

In [ ]:
grader.check("q8")

In [ ]:
# Calificacion automatica — ejecuta al terminar
grader.check_all()